In [1]:
# ====== Setup: 找 repo root & 載入你的 modeling ======
import os, sys, pathlib, numpy as np, pandas as pd

def add_project_root(marker_folder="project_package", max_up=8):
    cur = pathlib.Path(".").resolve()
    for _ in range(max_up + 1):
        if (cur / marker_folder).is_dir():
            if str(cur) not in sys.path:
                sys.path.insert(0, str(cur))
            return cur
        cur = cur.parent
    raise RuntimeError(f"Could not find '{marker_folder}'")

PROJECT_ROOT = add_project_root()
print("PROJECT_ROOT =", PROJECT_ROOT)

from project_package.modeling import (
    load_csv_dedup, make_binary_target, EXCLUDE_ALWAYS
)

CSV_NAME = "ncr_ride_bookings_with_weather_filled_scaled_short.csv"
csv_path = None
for p in [PROJECT_ROOT/CSV_NAME, PROJECT_ROOT/"datasets"/CSV_NAME, pathlib.Path(CSV_NAME)]:
    if p.exists():
        csv_path = p
        break
assert csv_path is not None, "CSV not found."

df = load_csv_dedup(str(csv_path))
df, target_col = make_binary_target(df)

# ====== 1) 以「關鍵字」擴充外洩欄位：含義等同但命名不同的欄位一併踢掉 ======
LEAKAGE_KEYWORDS = [
    # 結果/取消/不完成
    "status", "cancel", "cancellation", "incomplete",
    # 事後才知道的行程數值
    "vtat", "ctat", "booking value", "ride distance", "rating",
    # 事後的下車資訊/天氣/座標（dropoff）
    "dropoff", "drop_", "drop latitude", "drop longitude",
    # 任何 *_fill 跟 *_scaled 的變體
    "_fill", "_scaled",
]

def build_leakage_set(df):
    keep = set(EXCLUDE_ALWAYS)  # 你原本的黑名單
    cols = []
    for c in df.columns:
        lc = c.lower()
        if any(k in lc for k in LEAKAGE_KEYWORDS):
            cols.append(c)
    return keep | set(cols)

LEAK_SET = build_leakage_set(df)
print("Leakage-like columns to drop (preview 40):", list(sorted(LEAK_SET))[:40], "...", f"total={len(LEAK_SET)}")

# ====== 2) 只用「下單時可得」的特徵重訓 & 評估 ======
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

def feat_types(X, max_cat=200):
    num, cat = [], []
    for c in X.columns:
        s = X[c]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            num.append(c)
        elif pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
            if s.nunique(dropna=False) <= max_cat:
                cat.append(c)
    return num, cat

def make_preprocessor(X):
    num, cat = feat_types(X)
    return ColumnTransformer(
        [
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
            ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                              ("ohe", OneHotEncoder(handle_unknown="ignore"))]), cat),
        ],
        remainder="drop",
        sparse_threshold=1.0,
    )

# 時序切分（不用 ID 群組，下一段 B 會處理）
def time_split(df, target, ratio=0.2, time_col="booking_datetime"):
    d = df.copy()
    if time_col in d.columns:
        d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
        d = d.sort_values(time_col).reset_index(drop=True)
    cut = int(len(d) * (1 - ratio))
    return d.iloc[:cut].copy(), d.iloc[cut:].copy()

def run_once(df, title):
    drop_cols = set([target_col]) | LEAK_SET
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")
    y = df[target_col].astype(int).values
    tr, va = time_split(df, target_col, ratio=0.2)
    Xtr = tr.drop(columns=[c for c in drop_cols if c in tr.columns], errors="ignore")
    ytr = tr[target_col].astype(int).values
    Xva = va.drop(columns=[c for c in drop_cols if c in va.columns], errors="ignore")
    yva = va[target_col].astype(int).values

    pre = make_preprocessor(Xtr)
    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
    pipe = Pipeline([("pre", pre), ("rf", clf)])
    pipe.fit(Xtr, ytr)
    yhat = pipe.predict(Xva)
    proba = pipe.predict_proba(Xva)[:,1]
    print(f"\n==== {title} (with strengthened leakage guard) ====")
    print(classification_report(yva, yhat, digits=4))
    print("ROC AUC:", round(roc_auc_score(yva, proba), 4))

run_once(df, "Time split")


# ====== Group-aware time split with gap ======
def group_time_split_with_gap(df, target, group_col, time_col="booking_datetime",
                              valid_ratio=0.2, gap_ratio=0.05):
    d = df.copy()
    if time_col in d.columns:
        d[time_col] = pd.to_datetime(d[time_col], errors="coerce")
        d = d.sort_values(time_col).reset_index(drop=True)
    n = len(d)
    n_valid = int(n * valid_ratio)
    n_gap = int(n * gap_ratio)
    cut_valid = n - n_valid
    cut_train = max(0, cut_valid - n_gap)

    # 初步切分：train(<=cut_train-1), gap(cut_train..cut_valid-1), valid(>=cut_valid)
    d_train = d.iloc[:cut_train].copy()
    d_valid = d.iloc[cut_valid:].copy()

    # 把「跨兩邊的 group」整個移到 valid（或整個丟掉 gap 也可）
    if group_col in d.columns:
        train_groups = set(d_train[group_col].astype(str))
        valid_groups = set(d_valid[group_col].astype(str))
        overlap = train_groups & valid_groups
        if overlap:
            mask_overlap = d_train[group_col].astype(str).isin(overlap)
            # 移到 valid 端
            d_valid = pd.concat([d_valid, d_train[mask_overlap]], ignore_index=True)
            d_train = d_train[~mask_overlap].copy()

    return d_train, d_valid

# ====== 以強化後的 LEAK_SET + group split 重新評估 ======
def run_group_time_split(df, title, group_col):
    drop_cols = set([target_col]) | LEAK_SET | set([group_col])
    tr, va = group_time_split_with_gap(df, target_col, group_col=group_col,
                                       time_col="booking_datetime", valid_ratio=0.2, gap_ratio=0.05)

    Xtr = tr.drop(columns=[c for c in drop_cols if c in tr.columns], errors="ignore")
    ytr = tr[target_col].astype(int).values
    Xva = va.drop(columns=[c for c in drop_cols if c in va.columns], errors="ignore")
    yva = va[target_col].astype(int).values

    pre = make_preprocessor(Xtr)
    clf = RandomForestClassifier(n_estimators=300, class_weight="balanced", n_jobs=-1, random_state=42)
    pipe = Pipeline([("pre", pre), ("rf", clf)])
    pipe.fit(Xtr, ytr)
    yhat = pipe.predict(Xva)
    proba = pipe.predict_proba(Xva)[:,1]

    print(f"\n==== {title} (Group time split + gap, drop group col) ====")
    print("Train/Valid sizes:", len(tr), len(va))
    print("Unique groups train/valid:", tr[group_col].nunique(), va[group_col].nunique())
    print("Group overlap after split:",
          len(set(tr[group_col].astype(str)) & set(va[group_col].astype(str))))
    print(classification_report(yva, yhat, digits=4))
    print("ROC AUC:", round(roc_auc_score(yva, proba), 4))

# 跑 Booking ID 群組切分
if "Booking ID" in df.columns:
    run_group_time_split(df, "Booking ID grouping", group_col="Booking ID")

# 補跑 Customer ID 群組切分（看你們商業定義想避免哪種重疊）
if "Customer ID" in df.columns:
    run_group_time_split(df, "Customer ID grouping", group_col="Customer ID")


PROJECT_ROOT = C:\Users\yauli\OneDrive\桌面\MADS-MS2-Uber-Analysis
Leakage-like columns to drop (preview 40): ['Avg CTAT_fill_scaled', 'Avg VTAT_fill_scaled', 'Booking Status', 'Booking Value_fill_scaled', 'BookingValue_missing_flag', 'CTAT_missing_flag', 'Cancelled Rides by Customer_fill', 'Cancelled Rides by Driver_fill', 'Customer Rating_fill', 'Driver Cancellation Reason_fill', 'Driver Ratings_fill', 'Incomplete Rides Reason_fill', 'Incomplete Rides_fill', 'Reason for cancelling by Customer_fill', 'Ride Distance_fill_scaled', 'VTAT_missing_flag', 'apparent_temperature_dropoff_scaled', 'apparent_temperature_dropoff_scaled.1', 'apparent_temperature_scaled', 'apparent_temperature_scaled.1', 'dew_point_2m_dropoff_scaled', 'dew_point_2m_dropoff_scaled.1', 'dew_point_2m_scaled', 'dew_point_2m_scaled.1', 'drop_address', 'drop_latitude', 'drop_locality', 'drop_longitude', 'drop_region', 'drop_station_latitude', 'drop_station_longitude', 'precipitation_dropoff_log_scaled', 'precipitation_log_